In [ ]:
!pip install -q nevergrad
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh ames

In [ ]:
import warnings

import matplotlib.pyplot as plt
import nevergrad
import numpy as np
import pandas as pd
import scipy.stats
import seaborn as sns
import sklearn.base
import sklearn.compose
import sklearn.inspection
import sklearn.linear_model
import sklearn.model_selection
import sklearn.pipeline
import sklearn.preprocessing
from IPython.display import display

# Nevergrad calls SciPy's COBYLA with a tolerance that SciPy corrects.
warnings.filterwarnings("ignore", message="COBYLA: Invalid RHOEND")

# Linear Regression

## Univariate Linear Regression

### Introduction

Linear regression is very interesting to code yourself: it is not too complex but allows you to touch on many machine learning concepts.

We will continue our work on the Ames (Iowa) house price data. Let's load it:

In [ ]:
train_df = pd.read_csv("ames/train.csv", index_col="Id")

In [ ]:
display(train_df)

### Columns Used

In this first part, we'll only use one feature, `GrLivArea` (the living area, in square feet), and `SalePrice` (the sale price, in dollars) as the target.

- *Plot `SalePrice` against `GrLivArea` with `sns.jointplot`.*
- *Standardize both columns with a `StandardScaler` each, $\mathbf{x}_{norm} = \frac{\mathbf{x} - \bar{\mathbf{x}}}{\sigma_\mathbf{x}}$, into two one-dimensional NumPy vectors, `x` and `y`. Keep the scalers in `x_scaler` and `y_scaler`: they'll serve to go back to the original units.*

**Useful documentation**

- [`seaborn.jointplot`](https://seaborn.pydata.org/generated/seaborn.jointplot.html)
- [`sklearn.preprocessing.StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)

In [ ]:
# Your code here

#### Additional hints

- `sns.jointplot(x="GrLivArea", y="SalePrice", data=train_df)`: the scatter plot and the distribution of each column.
- A `StandardScaler` expects a two-dimensional array, one column per variable: `train_df[["GrLivArea"]].to_numpy()`, with two pairs of brackets. `fit_transform` returns a column, which `[:, 0]` turns into a vector.
- Expected values: `x.shape == (1460,)`, `x.mean()` ≈ 0 and `x.std()` ≈ 1.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
sns.jointplot(x=..., y=..., data=train_df)
plt.show()

x_scaler = sklearn.preprocessing.StandardScaler()
x = x_scaler.fit_transform(train_df[[...]].to_numpy())[:, 0]

y_scaler = ...
y = ...
```

#### Solution

In [ ]:
sns.jointplot(x="GrLivArea", y="SalePrice", data=train_df)
plt.show()

# Standardize x
x_scaler = sklearn.preprocessing.StandardScaler()
x = x_scaler.fit_transform(train_df[["GrLivArea"]].to_numpy())[:, 0]

# Standardize y
y_scaler = sklearn.preprocessing.StandardScaler()
y = y_scaler.fit_transform(train_df[["SalePrice"]].to_numpy())[:, 0]

### Linear Regression Example

To understand the joint evolution of a house's sale price and living area, we'll restrict ourselves to a strong hypothesis: we can find an affine function that correctly models the relationship between inputs and outputs. As a reminder, an affine function has the form $f(x) = ax + b$.

Geometrically, this function can be represented by a line.

For instance, here, we'd like to find the following hypothesis:

In [ ]:
sns.regplot(x=x, y=y, scatter_kws=dict(alpha=0.10), line_kws=dict(color="red"))
plt.show()

All this is promising, but here we have used [`seaborn`](https://seaborn.pydata.org/) to calculate the parameters of the hypothesis that best represents the data. We will now see how to do this ourselves.

### Estimating the Parameters

To estimate $a$ and $b$, we need 2 things:

- a cost function, which tells us, for given parameters $a$ and $b$, whether we're doing well or not
- a method to estimate the best parameters given this cost function

### Predictions

As a prerequisite to computing the cost function, we need to know how to compute predictions.

To apply the linear transformation defined by $a$ and $b$ to $\mathbf{x}$, we simply use the multiplication and addition of scalars to a vector: $a\mathbf{x} + b$.

*Define the `predict` function so that it computes the linear transformation given by $a$ and $b$.*

In [ ]:
def predict(x: np.ndarray, a: float, b: float) -> np.ndarray:
  # Your code here
  ...

#### Additional hints

- NumPy applies `a * x + b` to each element of the vector `x`: no loop.
- Example to check: `predict(np.array([1, 2, 3]), 3, 2)` is `[5, 8, 11]`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def predict(x: np.ndarray, a: float, b: float) -> np.ndarray:
  return ...
```

#### Solution

In [ ]:
def predict(x: np.ndarray, a: float, b: float) -> np.ndarray:
  return a * x + b


x_example = np.array([1, 2, 3])
y_example = np.array([4, 10, 3])
a_example = 3
b_example = 2

print(predict(x_example, a_example, b_example))

### Cost Function

The most widely used cost function is the mean of the squared differences between the hypothesis's predictions, with the current parameters, and the actual values, divided by 2 (the $\frac{1}{2}$ simplifies the derivative).

Denoting $E$ the cost function and $r$ the number of training examples, we can write:

$$
  E(a, b) = \frac{1}{2r}\sum_{i=1}^r (a x_i + b - y_i)^2
$$

- *Define the `residuals` function, which returns the vector of the differences $f(x_i) - y_i$, using the `predict` function.*
- *Define the `cost` function using the `residuals` function.*

In [ ]:
def residuals(x: np.ndarray, y: np.ndarray, a: float, b: float) -> np.ndarray:
  # Your code here
  ...


def cost(x: np.ndarray, y: np.ndarray, a: float, b: float) -> float:
  # Your code here
  ...

#### Additional hints

- `np.mean(v ** 2)`: the mean of the squares of `v`'s elements.
- Example to check, with `x_example`, `y_example`, $a = 3$ and $b = 2$: residuals `[1, -2, 8]`, cost `11.5`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def residuals(x: np.ndarray, y: np.ndarray, a: float, b: float) -> np.ndarray:
  return predict(...) - ...


def cost(x: np.ndarray, y: np.ndarray, a: float, b: float) -> float:
  return np.mean(...) / 2
```

#### Solution

In [ ]:
def residuals(x: np.ndarray, y: np.ndarray, a: float, b: float) -> np.ndarray:
  return predict(x, a, b) - y


print("Residuals:", residuals(x_example, y_example, a_example, b_example))


def cost(x: np.ndarray, y: np.ndarray, a: float, b: float) -> float:
  return np.mean(residuals(x, y, a, b) ** 2) / 2


print("Cost:", cost(x_example, y_example, a_example, b_example))

### Optimizing the Parameters

Optimizing the parameters means minimizing the cost function:

$$\min_{a, b} E(a, b)$$

For linear regression, there is an exact formula, least squares (what scikit-learn's `LinearRegression` computes, in the second part). We won't use it here, for two reasons: it's costly on very large datasets, and the method we're going to code also serves logistic regression and many other models (neural networks, gradient boosting…).

We'll use gradient descent. This method is iterative and takes successive steps in the direction opposite to the cost function's derivative. In the case of linear regression, it converges to the global optimum (the best solution).

In pseudo-code, with the partial derivatives of the course's "Example of Derivation — Linear Regression" slide, the algorithm is:

$$
\begin{aligned}
& \text{until convergence:} \\
& \quad a \leftarrow a - \eta \frac{\partial E}{\partial a} = a - \eta \frac{1}{r}\sum_{i=1}^r (a x_i + b - y_i)\, x_i \\
& \quad b \leftarrow b - \eta \frac{\partial E}{\partial b} = b - \eta \frac{1}{r}\sum_{i=1}^r (a x_i + b - y_i) \\
\end{aligned}
$$

where $\eta$ is the learning rate.

*Implement the `gradient` function, which returns the two partial derivatives $\frac{\partial E}{\partial a}$ and $\frac{\partial E}{\partial b}$, using the `residuals` function.*

In [ ]:
def gradient(x: np.ndarray, y: np.ndarray, a: float, b: float) -> tuple[float, float]:
  # Your code here
  ...

#### Additional hints

- With `res` the vector of residuals: $\frac{\partial E}{\partial a}$ is the mean of `res * x`, i.e. `(x @ res) / y.size` (`@`: the dot product), and $\frac{\partial E}{\partial b}$ the mean of `res`.
- Example to check: `gradient(x_example, y_example, 3, 2)` is `(7.0, 2.33…)`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def gradient(x: np.ndarray, y: np.ndarray, a: float, b: float) -> tuple[float, float]:
  res = residuals(x, y, a, b)
  g_a = ...
  g_b = ...
  return g_a, g_b
```

#### Solution

In [ ]:
def gradient(x: np.ndarray, y: np.ndarray, a: float, b: float) -> tuple[float, float]:
  res = residuals(x, y, a, b)
  g_a = (x @ res) / y.size
  g_b = res.mean()
  return g_a, g_b


g_a, g_b = gradient(x_example, y_example, a_example, b_example)
print(f"g_a = {g_a:.2f}, g_b = {g_b:.2f}")

### Gradient Descent

*Implement gradient descent in the `gradient_descent` function: start from randomly drawn $a$ and $b$, then, at most `nb_iter` times, update them with the step `eta` ($\eta$); stop as soon as the cost decreases by less than `epsilon` in one iteration. It returns $a$, $b$ and the list of successive costs, the initial cost included.*

In [ ]:
def gradient_descent(
  x: np.ndarray, y: np.ndarray, eta: float, nb_iter: int, epsilon: float = 1e-6
) -> tuple[float, float, list[float]]:
  # Your code here
  ...

#### Additional hints

- `rng = np.random.default_rng(0)` then `a, b = rng.random(2)`: a random draw, but the same on every run.
- `break` leaves the `for` loop as soon as `costs[-2] - costs[-1] < epsilon`.
- Expected values for `gradient_descent(x, y, 0.1, 1000)`: stopping at iteration 43, $a$ ≈ 0.71, $b$ ≈ 0 and a final cost ≈ 0.25.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def gradient_descent(
  x: np.ndarray, y: np.ndarray, eta: float, nb_iter: int, epsilon: float = 1e-6
) -> tuple[float, float, list[float]]:
  rng = np.random.default_rng(0)
  a, b = rng.random(2)
  costs = [cost(x, y, a, b)]
  for i in range(nb_iter):
    g_a, g_b = ...
    a -= ...
    b -= ...
    costs.append(...)
    if ...:
      break
  return a, b, costs
```

#### Solution

In [ ]:
def gradient_descent(
  x: np.ndarray, y: np.ndarray, eta: float, nb_iter: int, epsilon: float = 1e-6
) -> tuple[float, float, list[float]]:
  # A random draw, the same on every run
  rng = np.random.default_rng(0)
  a, b = rng.random(2)
  print(f"Initial (random) parameters: a = {a:.2f}, b = {b:.2f}")
  costs = [cost(x, y, a, b)]
  print(f"Initial cost: {costs[0]:.3f}")
  for i in range(nb_iter):
    g_a, g_b = gradient(x, y, a, b)
    a -= eta * g_a
    b -= eta * g_b
    costs.append(cost(x, y, a, b))
    if costs[-2] - costs[-1] < epsilon:
      print(f"Stopping: not enough progress at iteration {i}")
      break
  print(f"Final parameters: a = {a:.3f}, b = {b:.3f}")
  print(f"Final cost: {costs[-1]:.3f}")
  return a, b, costs

### Checking Our Hypothesis

Let's run gradient descent with a step $\eta = 0.1$ and at most 1,000 iterations: the successive costs should decrease.

In [ ]:
a, b, costs = gradient_descent(x, y, 0.1, 1000)
print("First costs:", np.round(costs[:5], 3))

Let's now check the learned model, brought back to the original units (dollars and square feet), with a few plots:

- predicted values against actual values
- residuals
- training costs

In [ ]:
inv_x = x_scaler.inverse_transform(x[:, None])[:, 0]
inv_y = y_scaler.inverse_transform(y[:, None])[:, 0]
inv_yh = y_scaler.inverse_transform(predict(x, a, b)[:, None])[:, 0]

xs = np.arange(0, 6_000, 100)[:, None]
ys = y_scaler.inverse_transform(predict(x_scaler.transform(xs), a, b))
plt.plot(inv_x, inv_y, "b.", alpha=0.10)
plt.plot(xs, ys, "r")
plt.title("Predictions and actual values")
plt.xlabel("Living area (square feet)")
plt.ylabel("Sale price ($)")
plt.show()

plt.plot(inv_x, inv_yh - inv_y, "r.", alpha=0.10)
plt.title("Residuals")
plt.xlabel("Living area (square feet)")
plt.ylabel("Residuals ($)")
plt.show()

plt.plot(range(len(costs)), costs)
plt.title("Cost during gradient descent")
plt.xlabel("Iterations")
plt.ylabel("Cost E")
plt.show()

### Correcting the Positive Skewness

The residuals spread more and more as the area grows, and the distribution of prices is skewed: a long tail towards expensive houses, which pull the line. This positive skewness of the output variable can be corrected by taking its logarithm, [`np.log1p`](https://numpy.org/doc/stable/reference/generated/numpy.log1p.html) ($\log(1 + y)$), which tightens the tail and brings the distribution closer to a normal one; its inverse transformation, [`np.expm1`](https://numpy.org/doc/stable/reference/generated/numpy.expm1.html), brings the predictions back to dollars.

The next cell compares the two distributions (with the closest normal distribution, in black), then defines `Y`, the standardized log-price.

In [ ]:
def plot_with_normal_fit(values: np.ndarray) -> None:
  values = np.ravel(values)
  sns.histplot(values, stat="density", kde=True)
  xs = np.linspace(values.min(), values.max(), 200)
  plt.plot(xs, scipy.stats.norm.pdf(xs, *scipy.stats.norm.fit(values)), "k")


plot_with_normal_fit(train_df["SalePrice"].to_numpy())
plt.title("Distribution of SalePrice")
plt.show()

Y_scaler_log = sklearn.preprocessing.StandardScaler()
Y = Y_scaler_log.fit_transform(np.log1p(train_df[["SalePrice"]].to_numpy()))[:, 0]

plot_with_normal_fit(Y)
plt.title("Distribution of log(1 + SalePrice), standardized")
plt.show()

Let's run gradient descent again on `Y`, then the same plots, in dollars:

In [ ]:
a, b, costs = gradient_descent(x, Y, 0.01, 5000)

In [ ]:
inv_Y = np.expm1(Y_scaler_log.inverse_transform(Y[:, None])[:, 0])
inv_Yh = np.expm1(Y_scaler_log.inverse_transform(predict(x, a, b)[:, None])[:, 0])
Ys = np.expm1(Y_scaler_log.inverse_transform(predict(x_scaler.transform(xs), a, b)))

plt.plot(inv_x, inv_Y, "b.", alpha=0.10)
plt.plot(xs, Ys, "r")
plt.title("Predictions and actual values")
plt.xlabel("Living area (square feet)")
plt.ylabel("Sale price ($)")
plt.show()

plt.plot(inv_x, inv_Yh - inv_Y, "r.", alpha=0.10)
plt.title("Residuals")
plt.xlabel("Living area (square feet)")
plt.ylabel("Residuals ($)")
plt.show()

plt.plot(range(len(costs)), costs)
plt.title("Cost during gradient descent")
plt.xlabel("Iterations")
plt.ylabel("Cost E")
plt.show()

## Multivariate Linear Regression

Let's move on to all the features of the houses: $f(\mathbf{x}) = w_1 x_1 + \dots + w_d x_d + b = \mathbf{w} \cdot \mathbf{x} + b$, one weight $w_j$ per variable instead of $a$ (the course's "Several Variables" slide).

The `preprocess` function performs a full preprocessing of the Ames data (no longer just the `GrLivArea` column): missing values filled, ordinal variables converted to integers, categorical ones encoded as indicators (one-hot). It returns the `X_train` and `X_test` matrices, the prices `Y_train` (one column) and the names of the columns, `columns`.

In [ ]:
def preprocess(
  train_file: str, test_file: str
) -> tuple[np.ndarray, np.ndarray, np.ndarray, pd.Index]:
  train_X = pd.read_csv(train_file, index_col="Id")
  test_X = pd.read_csv(test_file, index_col="Id")

  train_Y = train_X[["SalePrice"]]
  train_X = train_X.drop(columns=["SalePrice"])

  all_X = pd.concat([train_X, test_X])

  # Fill with the median
  cols_1 = ["LotFrontage"]
  all_X[cols_1] = all_X[cols_1].fillna(train_X[cols_1].median())

  # Fill with the mode
  cols_2 = [
    "MSZoning",
    "Electrical",
    "KitchenQual",
    "Exterior1st",
    "Exterior2nd",
    "SaleType",
    "Utilities",
  ]
  all_X[cols_2] = all_X[cols_2].fillna(train_X[cols_2].mode().iloc[0, :])

  # Fill with 0
  cols_4 = [
    "GarageYrBlt",
    "GarageArea",
    "GarageCars",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtFullBath",
    "BsmtHalfBath",
    "BsmtUnfSF",
    "MasVnrArea",
    "TotalBsmtSF",
  ]
  all_X[cols_4] = all_X[cols_4].fillna(0)

  # Other fills
  cols_5 = ["Functional"]
  all_X[cols_5] = all_X[cols_5].fillna("Typ")

  # All the other NAs get the string NA, which will be a category
  # of its own
  all_X = all_X.fillna("NA")

  # The numerical code becomes a string, so that it's treated as a
  # categorical variable
  cols_numerical2label = ["MSSubClass"]
  all_X[cols_numerical2label] = all_X[cols_numerical2label].astype(str)

  quality_mapping = dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5)
  quality_columns = [
    "BsmtCond",
    "BsmtQual",
    "ExterCond",
    "ExterQual",
    "FireplaceQu",
    "GarageCond",
    "GarageQual",
    "HeatingQC",
    "KitchenQual",
    "PoolQC",
  ]
  street_mapping = dict(NA=0, Grvl=1, Pave=2)
  bsmt_fin_mapping = dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6)

  ordinal_mapping = dict(
    Alley=street_mapping,
    BsmtExposure=dict(NA=0, No=1, Mn=2, Av=3, Gd=4),
    BsmtFinType1=bsmt_fin_mapping,
    BsmtFinType2=bsmt_fin_mapping,
    Functional=dict(Sal=1, Sev=2, Maj2=3, Maj1=4, Mod=5, Min2=6, Min1=7, Typ=8),
    LandSlope=dict(Sev=1, Mod=2, Gtl=3),
    LotShape=dict(IR3=1, IR2=2, IR1=3, Reg=4),
    PavedDrive=dict(NA=0, N=1, P=2, Y=3),
    Street=dict(Grvl=1, Pave=2),
    Utilities=dict(ELO=1, NoSeWa=2, NoSewr=3, AllPub=4),
  )

  for quality_column in quality_columns:
    ordinal_mapping[quality_column] = quality_mapping

  # The ordinal variables become integers (a value missing from the
  # mapping would give a NaN, and astype(int) would raise an error)
  for column, mapping in ordinal_mapping.items():
    all_X[column] = all_X[column].map(mapping).astype(int)

  print(f"Number of NAs: {all_X.isnull().sum().sum()}")

  dummies = pd.get_dummies(all_X)
  return (
    dummies.iloc[: train_X.shape[0], :].to_numpy(dtype=float),
    train_Y.to_numpy(dtype=float),
    dummies.iloc[train_X.shape[0] :, :].to_numpy(dtype=float),
    dummies.columns,
  )

In [ ]:
X_train, Y_train, X_test, columns = preprocess("ames/train.csv", "ames/test.csv")
X_train.shape

### Normalizing the Output Variable

*Standardize the prices `Y_train` with a `StandardScaler`, kept in `Y_scaler`, into a new variable `Y_train_scaled`.*

**Useful documentation**

- [`sklearn.preprocessing.StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)

In [ ]:
# Your code here

#### Additional hints

- As for `y`, but `Y_train` already has two dimensions (one column): `fit_transform(Y_train)` directly.
- Expected values: `Y_train_scaled.shape == (1460, 1)`, with a mean ≈ 0.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
Y_scaler = ...
Y_train_scaled = ...
```

#### Solution

In [ ]:
Y_scaler = sklearn.preprocessing.StandardScaler()
Y_train_scaled = Y_scaler.fit_transform(Y_train)

### Training a Linear Regression

- *Train a `LinearRegression` (least squares) on `X_train` and `Y_train_scaled`.*
- *Display the 10 largest coefficients in absolute value with the name of their column. Why can't they be compared with each other as long as the features aren't standardized? (the course's "Several Variables" slide)*
- *Predict the price of the first two houses, in dollars, and compare it to their actual price.*

**Useful documentation**

- [`sklearn.linear_model.LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)

In [ ]:
# Your code here

#### Additional hints

- `linear_regression.coef_` has one row per target: `coef_[0]` for the only one here.
- `pd.Series(values, index=columns)` pairs each weight with its column; `.abs().sort_values(ascending=False).index[:10]` gives the 10 columns with the largest weight in absolute value.
- `Y_scaler.inverse_transform(...)` brings the predictions back to dollars.
- Expected values: the largest weights go to rare indicators, like `RoofMatl_ClyTile` (≈ −7.6); the first two houses, at $208,500 and $181,500, are predicted at ≈ $210,700 and $201,700.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
linear_regression = sklearn.linear_model.LinearRegression()
linear_regression.fit(..., ...)

coefficients = pd.Series(linear_regression.coef_[0], index=columns)
largest = ...
print(coefficients[largest])

predictions = Y_scaler.inverse_transform(linear_regression.predict(X_train[:2]))
print(predictions[:, 0], Y_train[:2, 0])
```

#### Solution

In [ ]:
linear_regression = sklearn.linear_model.LinearRegression()
linear_regression.fit(X_train, Y_train_scaled)

coefficients = pd.Series(linear_regression.coef_[0], index=columns)
largest = coefficients.abs().sort_values(ascending=False).index[:10]
print(coefficients[largest])

# A weight reads for an increase of 1 in its column: 1 square foot, 1 year,
# or an indicator going from 0 to 1. These units have nothing in common, and
# a rare indicator (a few houses) can get a huge weight: only the weights of
# standardized columns compare.

predictions = Y_scaler.inverse_transform(linear_regression.predict(X_train[:2]))
print("Predicted prices:", predictions[:, 0].round())
print("Actual prices:", Y_train[:2, 0])

### Cross-Validation

*Write a function `score(model, X, Y)` that returns the mean R² of a 5-fold cross-validation of the model `model` (`cross_val_score`, `scoring="r2"`), and evaluate it on a `LinearRegression`.*

The model is wrapped in a `TransformedTargetRegressor`: it standardizes the target in each fold, on that fold's training rows only, then brings the predictions back to dollars. So `score` takes the raw prices, `Y_train`, and no information from the validation rows enters the training. A linear model's R² doesn't change, but the `alpha` penalty of the regularized models, further on, then reads on the scale of a standardized target, as in `Y_train_scaled`.

**Useful documentation**

- [`sklearn.model_selection.cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)
- [`sklearn.compose.TransformedTargetRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.TransformedTargetRegressor.html)

In [ ]:
def score(
  model: sklearn.base.BaseEstimator,
  X: np.ndarray = X_train,
  Y: np.ndarray = Y_train,
) -> float:
  # Your code here
  ...

#### Additional hints

- `sklearn.compose.TransformedTargetRegressor(regressor=model, transformer=sklearn.preprocessing.StandardScaler())`.
- `cross_val_score(estimator, X, Y, cv=5, scoring="r2")` returns the 5 scores: their mean with `.mean()`.
- Expected value: an R² ≈ 0.80 for `LinearRegression`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def score(
  model: sklearn.base.BaseEstimator,
  X: np.ndarray = X_train,
  Y: np.ndarray = Y_train,
) -> float:
  standardized_target = sklearn.compose.TransformedTargetRegressor(
    regressor=..., transformer=...
  )
  scores = sklearn.model_selection.cross_val_score(...)
  return scores.mean()


print(score(sklearn.linear_model.LinearRegression()))
```

#### Solution

In [ ]:
def score(
  model: sklearn.base.BaseEstimator,
  X: np.ndarray = X_train,
  Y: np.ndarray = Y_train,
) -> float:
  # The target is standardized in each fold, on that fold's training rows
  # only.
  standardized_target = sklearn.compose.TransformedTargetRegressor(
    regressor=model, transformer=sklearn.preprocessing.StandardScaler()
  )
  scores = sklearn.model_selection.cross_val_score(
    standardized_target, X, Y, cv=5, scoring="r2"
  )
  return scores.mean()


print(f"LinearRegression R²: {score(sklearn.linear_model.LinearRegression()):.3f}")

### L1 and L2 Regularization

Regularized models add a penalty on the weights to the error (the course's "Regularization" slide): `Ridge` (L2, $E + \lambda \sum_j w_j^2$) shrinks the weights, `Lasso` (L1, $E + \lambda \sum_j \lvert w_j \rvert$) zeroes some of them. The course's $\lambda$ is scikit-learn's `alpha` parameter. The penalty applies to the weights, which depend on each column's unit: so the features are standardized first, in a pipeline (`make_pipeline(StandardScaler(), Lasso(alpha=...))`), which fits the `StandardScaler` on the training rows of each cross-validation fold only.

- *For `alpha` in `np.logspace(-3, 1, 5)`, compute the score of a standardized Lasso, `make_pipeline(StandardScaler(), Lasso(alpha=alpha))`, and of the standardized Ridge with the same `alpha`.*
- *Train each pipeline on `X_train` and `Y_train_scaled`, and count the zero weights of its model. What does `alpha` do to the Lasso? To the Ridge?*

**Useful documentation**

- [`sklearn.pipeline.make_pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
- [`sklearn.linear_model.Lasso`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html)
- [`sklearn.linear_model.Ridge`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html)

In [ ]:
# Your code here

#### Additional hints

- `pipeline[-1]`: the last step of the pipeline, the model; `pipeline[-1].coef_`: its weights.
- `np.sum(weights == 0)`: the number of zero weights.
- Expected values: the Lasso goes from 56 zero weights (out of 244) at `alpha = 0.001` to 152 at `0.01`, where its R² is the best (≈ 0.83), then to all of them from `1` on, where its R² drops to 0. The Ridge keeps an R² ≈ 0.81 and a single zero weight, that of a column constant on the training set.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
for alpha in np.logspace(-3, 1, 5):
  lasso = sklearn.pipeline.make_pipeline(
    sklearn.preprocessing.StandardScaler(), sklearn.linear_model.Lasso(alpha=alpha)
  )
  ridge = ...
  lasso.fit(X_train, Y_train_scaled)
  ridge.fit(...)
  print(alpha, score(lasso), np.sum(lasso[-1].coef_ == 0), ...)
```

#### Solution

In [ ]:
for alpha in np.logspace(-3, 1, 5):
  lasso = sklearn.pipeline.make_pipeline(
    sklearn.preprocessing.StandardScaler(), sklearn.linear_model.Lasso(alpha=alpha)
  )
  ridge = sklearn.pipeline.make_pipeline(
    sklearn.preprocessing.StandardScaler(), sklearn.linear_model.Ridge(alpha=alpha)
  )
  lasso.fit(X_train, Y_train_scaled)
  ridge.fit(X_train, Y_train_scaled)
  print(
    f"alpha = {alpha:g}: "
    f"Lasso R² = {score(lasso):.3f}, zero weights = {np.sum(lasso[-1].coef_ == 0)}; "
    f"Ridge R² = {score(ridge):.3f}, zero weights = {np.sum(ridge[-1].coef_ == 0)}"
  )

### Hyperparameter Search

scikit-learn searches hyperparameters on a grid (`GridSearchCV`), at random (`RandomizedSearchCV`) or [by successive halving](https://scikit-learn.org/stable/modules/grid_search.html) (`HalvingGridSearchCV`, `HalvingRandomSearchCV`, still experimental), which discards poor candidates early. Third-party libraries explore the space more finely, taking the previous trials into account: [Optuna](https://optuna.org/), the most widespread, or [Nevergrad](https://facebookresearch.github.io/nevergrad/), from Meta, which we use here. As `alpha` varies over several orders of magnitude, it is searched on a logarithmic scale.

- *With `GridSearchCV`, search for the best `alpha` of the standardized Ridge (the pipeline of the previous exercise) among `np.logspace(-3, 3, 13)`.*
- *Write a function `loss(alpha, X=X_train)`, the opposite of this Ridge's score on `X`: it is minimal for the best model.*
- *Minimize `loss` with Nevergrad, `alpha` on a logarithmic scale between $10^{-3}$ and $10^3$, and keep in `best_model` the standardized Ridge with the best `alpha`, `best_alpha`.*

**Useful documentation**

- [`sklearn.model_selection.GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
- [`nevergrad.p.Log`](https://facebookresearch.github.io/nevergrad/parametrization.html)
- [`nevergrad.optimizers.NGOpt`](https://facebookresearch.github.io/nevergrad/optimization.html)

In [ ]:
# Your code here

#### Additional hints

- `make_pipeline` names each step after its class, in lowercase: the `alpha` parameter of the `ridge` step is called `ridge__alpha` in the grid, `dict(ridge__alpha=...)`. After `search.fit(X_train, Y_train_scaled)`: `search.best_params_` and `search.best_score_`.
- Nevergrad: `nevergrad.p.Instrumentation(alpha=nevergrad.p.Log(lower=1e-3, upper=1e3))` describes the parameter, `nevergrad.optimizers.NGOpt(parametrization=..., budget=50)` tries 50 values, `optimizer.minimize(loss)` returns the best one, in `recommendation.kwargs["alpha"]`.
- Expected values: `alpha` ≈ 316 for `GridSearchCV` and ≈ 450 for Nevergrad, with an R² ≈ 0.84 (against ≈ 0.81 at `alpha = 1`).

A skeleton to copy into the exercise cell, replacing each `...`:

```python
ridge = sklearn.pipeline.make_pipeline(
  sklearn.preprocessing.StandardScaler(), sklearn.linear_model.Ridge()
)
search = sklearn.model_selection.GridSearchCV(
  ridge, dict(ridge__alpha=...), scoring="r2", cv=5
)
search.fit(X_train, Y_train_scaled)
print(search.best_params_, search.best_score_)


def loss(alpha: float, X: np.ndarray = X_train) -> float:
  ridge = ...
  return -score(ridge, X=X)


parametrization = nevergrad.p.Instrumentation(alpha=nevergrad.p.Log(...))
optimizer = nevergrad.optimizers.NGOpt(parametrization=parametrization, budget=50)
recommendation = optimizer.minimize(loss)
best_alpha = recommendation.kwargs["alpha"]
best_model = ...
```

#### Solution

In [ ]:
ridge = sklearn.pipeline.make_pipeline(
  sklearn.preprocessing.StandardScaler(), sklearn.linear_model.Ridge()
)
search = sklearn.model_selection.GridSearchCV(
  ridge, dict(ridge__alpha=np.logspace(-3, 3, 13)), scoring="r2", cv=5
)
search.fit(X_train, Y_train_scaled)
grid_alpha = search.best_params_["ridge__alpha"]
print(f"GridSearchCV: alpha = {grid_alpha:.1f}, R² = {search.best_score_:.3f}")


def loss(alpha: float, X: np.ndarray = X_train) -> float:
  ridge = sklearn.pipeline.make_pipeline(
    sklearn.preprocessing.StandardScaler(), sklearn.linear_model.Ridge(alpha=alpha)
  )
  return -score(ridge, X=X)


parametrization = nevergrad.p.Instrumentation(
  alpha=nevergrad.p.Log(lower=1e-3, upper=1e3)
)
parametrization.random_state = np.random.RandomState(0)
optimizer = nevergrad.optimizers.NGOpt(parametrization=parametrization, budget=50)
recommendation = optimizer.minimize(loss, verbosity=0)
best_alpha = recommendation.kwargs["alpha"]
print(f"Nevergrad: alpha = {best_alpha:.1f}, R² = {-loss(best_alpha):.3f}")

best_model = sklearn.pipeline.make_pipeline(
  sklearn.preprocessing.StandardScaler(), sklearn.linear_model.Ridge(alpha=best_alpha)
)

### Feature Importance

A column's permutation importance is the drop in score when its values are shuffled, which breaks its link with the target. Computed on the training rows, it also measures what the model memorized: it is computed on a validation part, set aside during training.

- *Split `X_train` and `Y_train_scaled` into a training part (75%) and a validation part (25%), and train `best_model` on the first one.*
- *Compute the permutation importance of each column on the validation part, with 5 repeats, then plot the 10 largest.*

**Useful documentation**

- [`sklearn.model_selection.train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [`sklearn.inspection.permutation_importance`](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html)

In [ ]:
# Your code here

#### Additional hints

- `train_test_split(X_train, Y_train_scaled, test_size=0.25, random_state=0)` returns `X_fit, X_val, Y_fit, Y_val`.
- `permutation_importance(best_model, X_val, Y_val, n_repeats=5, n_jobs=-1, random_state=0)`: `n_jobs=-1` spreads the computation over all the cores; the result has an `importances_mean` attribute.
- `pd.Series(..., index=columns).nlargest(10).plot.bar()`.
- Expected values: `OverallQual` first, then neighborhoods (`Neighborhood_NridgHt`, `Neighborhood_NoRidge`), `KitchenQual` and `GrLivArea`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
X_fit, X_val, Y_fit, Y_val = sklearn.model_selection.train_test_split(...)
best_model.fit(X_fit, Y_fit)
feature_importances = sklearn.inspection.permutation_importance(
  best_model, ..., ..., n_repeats=5, n_jobs=-1, random_state=0
)
importances = pd.Series(feature_importances.importances_mean, index=columns)
importances.nlargest(10).plot.bar()
plt.show()
```

#### Solution

In [ ]:
X_fit, X_val, Y_fit, Y_val = sklearn.model_selection.train_test_split(
  X_train, Y_train_scaled, test_size=0.25, random_state=0
)
best_model.fit(X_fit, Y_fit)
feature_importances = sklearn.inspection.permutation_importance(
  best_model, X_val, Y_val, n_repeats=5, n_jobs=-1, random_state=0
)
importances = pd.Series(feature_importances.importances_mean, index=columns)
importances.nlargest(10).plot.bar()
plt.title("Permutation importance (validation part)")
plt.ylabel("Mean decrease in R²")
plt.show()

### Polynomial Features

`PolynomialFeatures` of degree 2 adds to columns their squares and their pairwise products: the model stays linear in its weights (the course's "A Linear Regression in Disguise" slide), but can follow a curvature or an interaction between two features (quality and area, for instance). With 244 columns, that would make nearly 30,000: we restrict ourselves to the 10 most important ones.

- *Restrict `X_train` and `X_test` to the 10 most important columns, then extend them with `PolynomialFeatures` of degree 2 into `X_train_poly` and `X_test_poly`.*
- *On `X_train_poly`, search with `GridSearchCV` for the best `alpha` of an L1 (`Lasso`), an L2 (`Ridge`) and a combined (`ElasticNet`) model, standardized, among `np.logspace(-3, 2, 11)`.*
- *Also tune the Ridge's `alpha` with Nevergrad, through `loss` on `X_train_poly`. Compare with the Ridge on the 10 columns alone and on all the columns: what do the squares and products bring?*

**Useful documentation**

- [`sklearn.preprocessing.PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)
- [`sklearn.linear_model.ElasticNet`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html)

In [ ]:
# Your code here

#### Additional hints

- `np.argsort(-feature_importances.importances_mean)[:10]`: the indices of the 10 most important columns; `X_train[:, top10]` selects them.
- `PolynomialFeatures(2, include_bias=False)`: without the constant column, which the model's intercept replaces. `fit_transform` on the training set, `transform` on the test set.
- A `Pipeline` with named steps, `[("scaler", StandardScaler()), ("regressor", regressor)]`, gives the same parameter name, `regressor__alpha`, to the three models. `Lasso(max_iter=10_000)` and `ElasticNet(max_iter=10_000)`: on these highly correlated columns, the default 1,000 iterations aren't enough to converge.
- Expected values: 65 columns; an R² ≈ 0.85 for the three models, against ≈ 0.80 on the 10 columns alone and ≈ 0.84 on all of them.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
top10 = np.argsort(-feature_importances.importances_mean)[:10]
polynomial_features = sklearn.preprocessing.PolynomialFeatures(2, include_bias=False)
X_train_poly = polynomial_features.fit_transform(...)
X_test_poly = polynomial_features.transform(...)

for regressor in (sklearn.linear_model.Lasso(max_iter=10_000), ..., ...):
  model = sklearn.pipeline.Pipeline(
    [("scaler", sklearn.preprocessing.StandardScaler()), ("regressor", regressor)]
  )
  search = sklearn.model_selection.GridSearchCV(
    model, dict(regressor__alpha=...), scoring="r2", cv=5
  )
  search.fit(X_train_poly, Y_train_scaled)
  print(type(regressor).__name__, search.best_params_, search.best_score_)


def loss_poly(alpha: float) -> float:
  return loss(alpha, X=...)
```

#### Solution

In [ ]:
top10 = np.argsort(-feature_importances.importances_mean)[:10]
polynomial_features = sklearn.preprocessing.PolynomialFeatures(2, include_bias=False)
X_train_poly = polynomial_features.fit_transform(X_train[:, top10])
X_test_poly = polynomial_features.transform(X_test[:, top10])
print(
  f"{X_train_poly.shape[1]} columns: the 10, their 10 squares and their 45 products"
)

In [ ]:
for regressor in (
  sklearn.linear_model.Lasso(max_iter=10_000),
  sklearn.linear_model.Ridge(),
  sklearn.linear_model.ElasticNet(max_iter=10_000),
):
  model = sklearn.pipeline.Pipeline(
    [("scaler", sklearn.preprocessing.StandardScaler()), ("regressor", regressor)]
  )
  search = sklearn.model_selection.GridSearchCV(
    model, dict(regressor__alpha=np.logspace(-3, 2, 11)), scoring="r2", cv=5
  )
  search.fit(X_train_poly, Y_train_scaled)
  alpha = search.best_params_["regressor__alpha"]
  name = type(regressor).__name__
  print(f"{name}: alpha = {alpha:g}, R² = {search.best_score_:.3f}")

In [ ]:
def loss_poly(alpha: float) -> float:
  return loss(alpha, X=X_train_poly)


parametrization = nevergrad.p.Instrumentation(
  alpha=nevergrad.p.Log(lower=1e-3, upper=1e3)
)
parametrization.random_state = np.random.RandomState(0)
optimizer = nevergrad.optimizers.NGOpt(parametrization=parametrization, budget=50)
recommendation = optimizer.minimize(loss_poly, verbosity=0)
alpha_poly = recommendation.kwargs["alpha"]
print(
  f"Polynomial Ridge, tuned by Nevergrad: alpha = {alpha_poly:.2f}, R² = {-loss_poly(alpha_poly):.3f}"
)
print(
  f"Ridge on the 10 columns alone: R² = {-loss(alpha_poly, X=X_train[:, top10]):.3f}"
)
print(f"Ridge on all the columns: R² = {-loss(best_alpha):.3f}")